# 🔄 RS-LiDAR: Đồng Bộ Dữ Liệu Sang Google Drive Cá Nhân Vĩnh Viễn

Notebook này phục vụ cho quy trình **sử dụng Google Drive Cá Nhân Vĩnh Viễn** để lưu trữ toàn bộ dữ liệu RS-LiDAR:
- **Thư mục chuẩn vĩnh viễn trên Drive cá nhân**: Đặt tên là **`RS-LiDAR`**.
- **Cơ chế hoạt động tối ưu**:
  1. **Tài khoản Colab hiện tại**: Đổi tên thư mục cũ trên Colab thành `RS-LiDAR-Old` (hoặc `RS-LiDAR-Local`), sau đó thêm lối tắt thư mục `RS-LiDAR` từ Drive cá nhân vào `My Drive` -> Notebook tự động nhận diện `RS-LiDAR-Old` là nguồn và copy toàn bộ dữ liệu sang `RS-LiDAR` (Drive cá nhân).
  2. **Các tài khoản Colab mới (từ tháng sau trở đi)**: Chỉ cần thêm lối tắt `RS-LiDAR` vào `My Drive` -> Mọi notebook (`LiDAR_Table2_Replication_Colab.ipynb`, v.v.) sẽ tự động nhận diện và chạy ngay mà **KHÔNG CẦN SỬA BẤT KỲ DÒNG CODE HAY ĐƯỜNG DẪN NÀO!**


## 1. Gắn Kết Google Drive Của Tài Khoản Colab Hiện Tại


In [ ]:
import os, sys, shutil, glob, time
from google.colab import drive

# Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = '/content/drive/My Drive' if os.path.exists('/content/drive/My Drive') else '/content/drive/MyDrive'

# Hàm kiểm tra số lượng dữ liệu thực nghiệm trong thư mục
def count_experiment_data(folder_path):
    if not os.path.exists(folder_path): return -1
    count = 0
    for sub in ['Lookahead_samples', 'Target_samples', 'test_results']:
        p = os.path.join(folder_path, sub)
        if os.path.exists(p):
            count += len(glob.glob(f'{p}/*'))
    return count

# Danh sách các tên thư mục ứng viên
candidates = [
    'RS-LiDAR-Old', 'RS-LiDAR-Local', 'RS-LiDAR-Previous',
    'RS-LiDAR', 'RS-LiDAR-Personal', 'RS-LiDAR-Backup', 'RS-LiDAR (1)'
]

existing_candidates = [c for c in candidates if os.path.exists(f'{base_drive}/{c}')]

SRC_DIR = None
DEST_DIR = None

# Trường hợp 1: Người dùng đã đổi tên thư mục cũ thành RS-LiDAR-Old / Local
# và thêm lối tắt Drive cá nhân là RS-LiDAR
for old_name in ['RS-LiDAR-Old', 'RS-LiDAR-Local', 'RS-LiDAR-Previous']:
    old_path = f'{base_drive}/{old_name}'
    if os.path.exists(old_path) and count_experiment_data(old_path) > 0:
        SRC_DIR = old_path
        DEST_DIR = f'{base_drive}/RS-LiDAR'
        break

# Trường hợp 2: Thư mục cũ vẫn giữ tên RS-LiDAR, còn lối tắt Drive cá nhân được đặt tên là RS-LiDAR-Personal / Backup
if SRC_DIR is None and os.path.exists(f'{base_drive}/RS-LiDAR'):
    for dest_name in ['RS-LiDAR-Personal', 'RS-LiDAR-Backup', 'RS-LiDAR (1)']:
        dest_path = f'{base_drive}/{dest_name}'
        if os.path.exists(dest_path):
            SRC_DIR = f'{base_drive}/RS-LiDAR'
            DEST_DIR = dest_path
            break

print('✅ Đã gắn kết Google Drive thành công!')
print('-' * 70)
if SRC_DIR and DEST_DIR and os.path.exists(DEST_DIR):
    print(f'📦 [NGUỒN] Thư mục chứa dữ liệu cũ cần sao lưu: {SRC_DIR}')
    print(f'🎯 [ĐÍCH]  Lối tắt dẫn về Drive cá nhân vĩnh viễn:  {DEST_DIR}')
    print('🚀 Sẵn sàng sao chép! Hãy chạy Cell bên dưới để bắt đầu.')
else:
    print('⚠️ CHƯA XÁC ĐỊNH ĐỦ NGUỒN VÀ ĐÍCH!')
    print('   Vui lòng làm theo hướng dẫn ở Bước 2 bên dưới:')
    print('   - Đổi tên thư mục dữ liệu cũ trên Drive Colab thành: RS-LiDAR-Old')
    print('   - Thêm lối tắt thư mục RS-LiDAR từ Drive cá nhân vào My Drive.')
print('-' * 70)


---
## 🌟 BƯỚC 2: CHUYỂN DỮ LIỆU TỪ COLAB HIỆN TẠI SANG DRIVE CÁ NHÂN

> **Mục tiêu**: Đưa toàn bộ checkpoints, latents và ảnh mẫu từ tài khoản Colab hiện tại vào thư mục `RS-LiDAR` trên Google Drive cá nhân.
>
> **2 bước chuẩn bị siêu đơn giản trên giao diện Google Drive**:
>
> 1. **Chuẩn bị trên Colab hiện tại**:
>    - Mở Google Drive của tài khoản Colab hiện tại -> Vào **Drive của tôi (My Drive)**.
>    - Chuột phải vào thư mục `RS-LiDAR` cũ đang có dữ liệu -> Chọn **Đổi tên (Rename)** thành **`RS-LiDAR-Old`** (đây sẽ là thư mục **NGUỒN**).
>
> 2. **Tạo và kết nối thư mục trên Drive Cá Nhân**:
>    - Mở Google Drive **Cá Nhân** -> Tạo thư mục tên là **`RS-LiDAR`**.
>    - Chuột phải vào `RS-LiDAR` -> Chọn **Chia sẻ (Share)** -> Nhập Gmail tài khoản Colab hiện tại (quyền **Người chỉnh sửa / Editor**).
>    - Quay lại Google Drive của tài khoản Colab hiện tại -> Vào mục **"Được chia sẻ với tôi" (Shared with me)** -> Chuột phải vào thư mục `RS-LiDAR` vừa nhận được -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)** -> Chọn lưu tại **Drive của tôi (My Drive)** (đây sẽ là thư mục **ĐÍCH** `RS-LiDAR`).


In [ ]:
# @title 🚀 BẮT ĐẦU ĐỒNG BỘ TOÀN BỘ DỮ LIỆU SANG DRIVE CÁ NHÂN
assert SRC_DIR and os.path.exists(SRC_DIR), f"❌ Không tìm thấy thư mục nguồn chứa dữ liệu cũ! Hãy kiểm tra lại thư mục RS-LiDAR-Old."
assert DEST_DIR and os.path.exists(DEST_DIR), f"❌ Không tìm thấy thư mục đích! Vui lòng làm theo hướng dẫn ở Bước 2: Thêm lối tắt RS-LiDAR từ Drive cá nhân vào My Drive."
assert os.path.abspath(SRC_DIR) != os.path.abspath(DEST_DIR), f"❌ Thư mục nguồn và đích trùng nhau ({SRC_DIR})! Vui lòng kiểm tra lại."

print("=" * 70)
print(f"🔄 BẮT ĐẦU SAO CHÉP DỮ LIỆU TỪ [{os.path.basename(SRC_DIR)}] SANG [{os.path.basename(DEST_DIR)}] (Drive Cá Nhân)...")
print("⚡ Quá trình diễn ra trực tiếp qua mạng nội bộ Google (siêu tốc, không tốn mạng cá nhân).")
print("=" * 70)

start_time = time.time()
copied_files = 0
total_bytes = 0

folders_to_sync = ["Lookahead_samples", "Target_samples", "test_results"]

for folder_name in folders_to_sync:
    src_folder = os.path.join(SRC_DIR, folder_name)
    dest_folder = os.path.join(DEST_DIR, folder_name)
    
    if not os.path.exists(src_folder):
        continue
        
    print(f"\n📦 Đang đồng bộ thư mục: {folder_name}...")
    os.makedirs(dest_folder, exist_ok=True)
    
    for root, dirs, files in os.walk(src_folder):
        rel_path = os.path.relpath(root, src_folder)
        target_root = os.path.join(dest_folder, rel_path)
        os.makedirs(target_root, exist_ok=True)
        
        for file in files:
            src_file = os.path.join(root, file)
            dest_file = os.path.join(target_root, file)
            
            # Chỉ copy nếu file đích chưa có hoặc kích thước khác nhau
            if not os.path.exists(dest_file) or os.path.getsize(src_file) != os.path.getsize(dest_file):
                shutil.copy2(src_file, dest_file)
                copied_files += 1
                total_bytes += os.path.getsize(src_file)
                if copied_files % 50 == 0:
                    print(f"   • Đã sao chép {copied_files} files ({total_bytes / (1024**2):.1f} MB)...", end="\r")

elapsed = time.time() - start_time
print("\n" + "=" * 70)
print(f"🎉 ĐỒNG BỘ THÀNH CÔNG SANG DRIVE CÁ NHÂN!")
print(f"📊 Tổng số file đã sao chép: {copied_files} files")
print(f"💾 Tổng dung lượng: {total_bytes / (1024**3):.2f} GB")
print(f"⏱️ Thời gian thực hiện: {elapsed:.1f} giây ({total_bytes / (1024**2) / max(elapsed, 0.1):.1f} MB/s)")
print(f"📍 Dữ liệu hiện đã an toàn 100% trên thư mục 'RS-LiDAR' ở Google Drive cá nhân của bạn!")
print(f"💡 Sau khi kiểm tra xong ở Bước 3, bạn có thể xóa thư mục '{os.path.basename(SRC_DIR)}' để giải phóng dung lượng cho tài khoản Colab này.")
print("=" * 70)


---
## 🌟 BƯỚC 3: KIỂM TRA TÍNH TOÀN VẸN TRÊN DRIVE CÁ NHÂN
Chạy cell này để kiểm tra xem trên Drive cá nhân đã đủ số lượng prompt và checkpoints chưa.


In [ ]:
# @title 🔍 KIỂM TRA TIẾN ĐỘ TRÊN DRIVE CÁ NHÂN
check_dir = DEST_DIR if (DEST_DIR and os.path.exists(DEST_DIR)) else f'{base_drive}/RS-LiDAR'
print(f"📊 BÁO CÁO DỮ LIỆU HIỆN CÓ TRÊN DRIVE CÁ NHÂN ({check_dir}):")
print("-" * 70)

# 1. Lookahead samples
look_folders = sorted(glob.glob(f"{check_dir}/Lookahead_samples/*"))
if look_folders:
    print(f"📁 [Lookahead_samples] Tìm thấy {len(look_folders)} thư mục:")
    for lf in look_folders:
        p_count = len(glob.glob(f"{lf}/[0-9]*/results.json"))
        lat_count = len(glob.glob(f"{lf}/[0-9]*/samples/latent.pt"))
        print(f"   • {os.path.basename(lf)}: {p_count}/553 prompts đã chấm reward | {lat_count}/553 latents có sẵn")
else:
    print("ℹ️ Chưa có thư mục Lookahead_samples.")

# 2. Target samples
targ_folders = sorted(glob.glob(f"{check_dir}/Target_samples/*"))
if targ_folders:
    print(f"\n📁 [Target_samples] Tìm thấy {len(targ_folders)} thí nghiệm đã sinh ảnh:")
    for tf in targ_folders:
        p_count = len(glob.glob(f"{tf}/[0-9]*/results.json"))
        has_geneval = os.path.exists(f"{tf}/geneval_summary.csv")
        has_pub = os.path.exists(f"{tf}/table2_publication_summary.csv")
        status_str = []
        if has_geneval: status_str.append("GenEval ✅")
        if has_pub: status_str.append("Table 2 ✅")
        extra = f" ({', '.join(status_str)})" if status_str else ""
        print(f"   • {os.path.basename(tf)}: {p_count}/553 prompts hoàn thành{extra}")
else:
    print("ℹ️ Chưa có thư mục Target_samples.")

print("-" * 70)
print("✅ MỌI THỨ ĐÃ SẴN SÀNG! DỮ LIỆU ĐÃ NẰM TRỌN VẸN TRÊN DRIVE CÁ NHÂN VĨNH VIỄN!")


---
## 💡 HƯỚNG DẪN DÀNH CHO CÁC TÀI KHOẢN COLAB MỚI (TỪ THÁNG SAU TRỞ ĐI)

Từ tháng sau, mỗi khi chuyển sang một **tài khoản Colab mới**, bạn **KHÔNG CẦN CHẠY LẠI NOTEBOOK BACKUP NÀY NỮA**, mà chỉ cần 3 bước cực nhanh:

1. **Trên Google Drive Cá Nhân**:
   - Chuột phải vào thư mục **`RS-LiDAR`** -> Chọn **Chia sẻ (Share)** -> Nhập Gmail của tài khoản Colab mới (quyền **Người chỉnh sửa / Editor**).

2. **Trên Google Drive của tài khoản Colab Mới**:
   - Mở Google Drive Colab mới -> Vào mục **"Được chia sẻ với tôi" (Shared with me)**.
   - Chuột phải vào thư mục **`RS-LiDAR`** -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)**.
   - Chọn lưu tại: **Drive của tôi (My Drive)** -> Bấm **Thêm (Add)**.
   - *(Lối tắt tạo ra sẽ tự động mang tên chính xác là **`RS-LiDAR`**)*.

3. **Mở Colab mới và chạy thực nghiệm ngay (KHÔNG CẦN SỬA CODE!)**:
   - Mở bất kỳ notebook nào: `LiDAR_Table2_Replication_Colab.ipynb` hoặc `LiDAR_Weaknesses_Colab.ipynb`.
   - Đường dẫn trong các notebook này mặc định là:
     ```python
     DRIVE_DIR = f"{base_drive}/RS-LiDAR"
     ```
   - Đường dẫn này sẽ **trỏ thẳng qua lối tắt vào đúng thư mục `RS-LiDAR` trên Drive cá nhân** của bạn!
   - Nhận diện 100% checkpoints, latents và ảnh mẫu cũ, tự động bỏ qua các prompt đã xong.
   - **HOÀN TOÀN KHÔNG CẦN CHỈNH SỬA BẤT KỲ DÒNG CODE NÀO!**
